<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:
try:
    import piplite
except ImportError:
    # In a standard Python environment the packages are installed outside
    # the notebook, so piplite is not required.
    piplite = None

if piplite is not None:
    await piplite.install(["folium", "pandas"])


In [ ]:
import folium
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [ ]:
# Download and read `spacex_launch_geo.csv`.
# JupyterLite uses JavaScript fetch; standard Jupyter can read the URL directly.
import io

URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"

try:
    from js import fetch
except ImportError:
    spacex_df = pd.read_csv(URL)
else:
    response = await fetch(URL)
    spacex_csv_file = io.BytesIO((await response.arrayBuffer()).to_py())
    spacex_df = pd.read_csv(spacex_csv_file)


Now, you can take a look at what are the coordinates for each site.


In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


### Solution

Create one circle and one text label for every unique launch site.


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Initialize a map that shows all launch sites in the United States.
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add a circle and a text label for every unique launch site.
for _, row in launch_sites_df.iterrows():
    coordinate = [row["Lat"], row["Long"]]

    circle = folium.Circle(
        location=coordinate,
        radius=1000,
        color="#000000",
        fill=True,
    ).add_child(folium.Popup(row["Launch Site"]))

    label = folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; color:#d35400; '
                'white-space:nowrap;"><b>%s</b></div>'
                % row["Launch Site"]
            ),
        ),
    )

    circle.add_to(site_map)
    label.add_to(site_map)

site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


### Explanation and findings

The loop reads each unique row in `launch_sites_df`. Folium expects every
location as `[latitude, longitude]`. `folium.Circle` highlights the launch pad,
the popup reveals its name when selected, and `DivIcon` keeps the site name
visible on the map.

All four launch sites are at relatively low northern latitudes, approximately
between 28°N and 35°N. They are therefore relatively close to the Equator on a
global scale, although none is directly on it. Lower-latitude locations can
benefit from the Earth's eastward rotation for suitable prograde launches.

All four sites are also close to a coastline. A coastal location provides an
open downrange corridor over the ocean and reduces exposure of populated areas
to launch failures or falling debris.

## Task 2: Mark the successful and failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:
spacex_df.tail(10)

Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()


### Solution

Map `class = 1` to green and `class = 0` to red. The column belongs in
`spacex_df` because this detailed dataframe contains the `class` value for
every launch.


In [ ]:
def assign_marker_color(launch_outcome):
    # Return green for a successful launch and red for a failed launch.
    return "green" if launch_outcome == 1 else "red"


spacex_df["marker_color"] = spacex_df["class"].apply(assign_marker_color)
spacex_df.tail(10)


### Solution

Add the cluster to the map, then add one color-coded marker for every launch
record.


In [ ]:
# Add the cluster layer to the current map.
site_map.add_child(marker_cluster)

# Add one marker for every launch. Multiple launches at the same coordinates
# are grouped automatically by MarkerCluster.
for _, record in spacex_df.iterrows():
    coordinate = [record["Lat"], record["Long"]]
    launch_result = "Success" if record["class"] == 1 else "Failure"

    marker = folium.Marker(
        location=coordinate,
        popup=f'{record["Launch Site"]}: {launch_result}',
        icon=folium.Icon(
            color="white",
            icon_color=record["marker_color"],
        ),
    )
    marker_cluster.add_child(marker)

site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [ ]:
# Quantify the visual comparison so the conclusion is reproducible.
success_by_site = (
    spacex_df.groupby("Launch Site", as_index=False)
    .agg(
        total_launches=("class", "count"),
        successful_launches=("class", "sum"),
        success_rate=("class", "mean"),
    )
)
success_by_site["success_rate"] = (
    success_by_site["success_rate"] * 100
).round(1)
success_by_site.sort_values("success_rate", ascending=False)

### Explanation and findings

`MarkerCluster` prevents the many launches sharing identical coordinates from
covering one another. Green icons represent successful landings and red icons
represent failures. In this dataset, **KSC LC-39A has the highest success rate
(76.9%)**, followed by CCAFS SLC-40 (42.9%), VAFB SLC-4E (40.0%), and CCAFS
LC-40 (26.9%). These percentages describe this historical sample, not a
guarantee about future launches.

## Task 3: Calculate the distances between a launch site and its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

### Solution

Use CCAFS SLC-40 as the origin and a coastline point selected with
`MousePosition`. Small coordinate differences are expected when selecting the
point manually.


In [ ]:
# CCAFS SLC-40 launch-site coordinates.
launch_site_lat = 28.563197
launch_site_lon = -80.576820

# A nearby coastline point selected from the interactive map.
coastline_lat = 28.56334
coastline_lon = -80.56799

distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon,
)

print(f"Distance to coastline: {distance_coastline:.2f} km")


In [ ]:
coastline_coordinate = [coastline_lat, coastline_lon]

distance_marker = folium.Marker(
    location=coastline_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size:12px; color:#d35400; '
            'white-space:nowrap;"><b>%s</b></div>'
            % f"Coastline: {distance_coastline:.2f} KM"
        ),
    ),
)
distance_marker.add_to(site_map)


### Solution

Connect the launch site and coastline point with a straight line.


In [ ]:
launch_site_coordinate = [launch_site_lat, launch_site_lon]
coordinates = [launch_site_coordinate, coastline_coordinate]

lines = folium.PolyLine(
    locations=coordinates,
    weight=2,
    color="blue",
    tooltip=f"Coastline: {distance_coastline:.2f} km",
)
site_map.add_child(lines)
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


### Solution

Repeat the distance calculation and map annotation for the closest selected
highway, railway, and city points.


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Nearby points selected from the base map using MousePosition.
proximities = {
    "Highway": [28.56335, -80.57085],
    "Railway": [28.57206, -80.58525],
    "City": [28.10473, -80.64531],
}

# Calculate the straight-line distance from CCAFS SLC-40 to each point.
distance_results = []
for proximity_name, proximity_coordinate in proximities.items():
    distance_km = calculate_distance(
        launch_site_lat,
        launch_site_lon,
        proximity_coordinate[0],
        proximity_coordinate[1],
    )
    distance_results.append(
        {"Proximity": proximity_name, "Distance (km)": round(distance_km, 2)}
    )

distance_summary = pd.DataFrame(distance_results)
distance_summary


In [ ]:
# Add a distance label and line for the highway, railway, and city points.
for result in distance_results:
    proximity_name = result["Proximity"]
    distance_km = result["Distance (km)"]
    proximity_coordinate = proximities[proximity_name]

    folium.Marker(
        location=proximity_coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; color:#d35400; '
                'white-space:nowrap;"><b>%s</b></div>'
                % f"{proximity_name}: {distance_km:.2f} KM"
            ),
        ),
    ).add_to(site_map)

    folium.PolyLine(
        locations=[launch_site_coordinate, proximity_coordinate],
        weight=2,
        color="blue",
        tooltip=f"{proximity_name}: {distance_km:.2f} km",
    ).add_to(site_map)


In [ ]:
display(distance_summary)
site_map


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### Final findings

For the analyzed **CCAFS SLC-40** site, the selected points are approximately
**0.58 km from the highway**, **1.28 km from the railway**, **0.86 km from the
coastline**, and **51.43 km from the city point**.

- The site is very close to road and rail infrastructure, which supports the
  movement of personnel, equipment, and heavy components.
- It is also very close to the coastline, providing an open trajectory over
  the ocean and reducing danger to populated areas.
- The selected city point is much farther away, consistent with keeping launch
  operations separated from dense population centers because of safety, noise,
  and falling-debris risks.

These are Haversine straight-line distances for one analyzed launch site. The
map suggests a similar location pattern for the other sites, but a rigorous
claim about every site would require repeating the distance calculations for
each one.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
